In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSV files
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Convert timestamp to timezone-aware datetimes in UTC then convert to Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
for df in (metric, trace, log_df, error_logs):
    if 'timestamp' in df.columns:
        df['ts_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Aggregate metric.csv by cmdb_id and kpi_name
metric_agg_full = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(count='count',
         p95=lambda x: x.quantile(0.95),
         mean='mean',
         min='min',
         max='max')
)

metric_agg_full.rename(columns={'kpi_name': 'name'}, inplace=True)
metric_agg_full['source'] = 'metric'
# Reorder columns
metric_agg_full = metric_agg_full[['source', 'cmdb_id', 'name', 'count', 'p95', 'mean', 'min', 'max']]

# Aggregate trace.csv by cmdb_id and trace_name
trace_agg_full = (
    trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(count='count',
         p95=lambda x: x.quantile(0.95),
         mean='mean',
         min='min',
         max='max')
)

trace_agg_full.rename(columns={'trace_name': 'name'}, inplace=True)
trace_agg_full['source'] = 'trace'
trace_agg_full = trace_agg_full[['source', 'cmdb_id', 'name', 'count', 'p95', 'mean', 'min', 'max']]

# Aggregate log.csv by cmdb_id and log_name
log_agg_full = (
    log_df
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .agg(count='count',
         p95=lambda x: x.quantile(0.95),
         mean='mean',
         min='min',
         max='max')
)

log_agg_full.rename(columns={'log_name': 'name'}, inplace=True)
log_agg_full['source'] = 'log'
log_agg_full = log_agg_full[['source', 'cmdb_id', 'name', 'count', 'p95', 'mean', 'min', 'max']]

# Combine aggregated tables
combined_agg_full = pd.concat([metric_agg_full, trace_agg_full, log_agg_full], ignore_index=True)

# Create compact top-20 table sorted by p95 descending (across all sources)
top20 = combined_agg_full.sort_values('p95', ascending=False).head(20).reset_index(drop=True)

# Per-file unique cmdb_id lists with counts
metric_cmdb_counts = metric.groupby('cmdb_id', as_index=False).size().rename(columns={'size':'count'}).sort_values('count', ascending=False).reset_index(drop=True)
trace_cmdb_counts = trace.groupby('cmdb_id', as_index=False).size().rename(columns={'size':'count'}).sort_values('count', ascending=False).reset_index(drop=True)
log_cmdb_counts = log_df.groupby('cmdb_id', as_index=False).size().rename(columns={'size':'count'}).sort_values('count', ascending=False).reset_index(drop=True)
error_cmdb_counts = error_logs.groupby('cmdb_id', as_index=False).size().rename(columns={'size':'count'}).sort_values('count', ascending=False).reset_index(drop=True)

# Display compact outputs: top-20 aggregated rows and per-file component lists with counts
top20, metric_cmdb_counts, trace_cmdb_counts, log_cmdb_counts, error_cmdb_counts

```
Out[1]:
```
summary = (
    "Summary of aggregation results:\n\n"
    "Top aggregated KPI groups (by P95):\n"
    "- The highest P95 values are dominated by metric data. The top entry is Redis01 "
    "OSLinux_NETWORK_ens160_NETPacketsIn with p95 ≈ 5.19e+09. Other very high p95 metric "
    "entries include Redis01 NETPacketsOut (~4.69e+09), Redis total_commands_processed (~4.56e+09), "
    "and Redis maxmemory (~4.29e+09). Most of the top-20 rows are from the metric source.\n\n"
    "Per-file component presence (unique cmdb_id counts and top components):\n"
    "- metric.csv: 29 unique cmdb_id values. Top counts: Mysql02 (4120), Mysql01 (4102), "
    "Redis01 (2256), Redis02 (2248), Tomcat04 (1655).\n"
    "- trace.csv: 12 unique cmdb_id values. Top counts: MG02 (1146), MG01 (1134), IG02 (450), "
    "IG01 (450), Tomcat01-04 (each 450).\n"
    "- log.csv: 7 unique cmdb_id values. Top counts: Tomcat02 (67), Tomcat04 (65), Tomcat03 (64), "
    "Tomcat01 (63), apache01 (62), apache02 (60), Mysql01 (8).\n"
    "- error_logs.csv: no cmdb_id entries (empty).\n\n"
    "Notes:\n"
    "- Full detailed aggregated tables are stored in kernel variables: metric_agg_full, trace_agg_full, "
    "log_agg_full, and combined_agg_full. Use those variables if you need the complete per-group results.\n"
    "- The compact top-20 table and the per-file component lists were displayed previously."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(    source   cmdb_id                                               name  count           p95          mean           min           max
0   metric   Redis01        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn     15  5.189691e+09  2.075790e+09  2.400000e+02  5.189787e+09
1   metric   Redis01       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut     15  4.691767e+09  1.876633e+09  1.950000e+02  4.691849e+09
2   metric   Redis01  redis-Redis_6379_Redis  (total_commands_proces...     31  4.561865e+09  4.561622e+09  4.561367e+09  4.561894e+09
3   metric   Redis01                redis-Redis_6379_Redis  (maxmemory)     31  4.294967e+09  4.294967e+09  4.294967e+09  4.294967e+09
4   metric   Redis02                redis-Redis_6379_Redis  (maxmemory)     31  4.294967e+09  4.294967e+09  4.294967e+09  4.294967e+09
5   metric  Tomcat01             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09
6   metric  Tomcat03           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09
7   metric  Tomcat01           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09
8   metric  Tomcat02           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09
9   metric  Tomcat03             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09
10  metric  Tomcat04           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09
11  metric  Tomcat04             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09
12  metric  Tomcat02             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09
13  metric   Redis02  redis-Redis_6379_Redis  (total_commands_proces...     31  3.698266e+09  3.698107e+09  3.697934e+09  3.698284e+09
14  metric   Redis01            redis-Redis_6379_Redis  (keyspace_hits)     31  3.451731e+09  3.451566e+09  3.451393e+09  3.451751e+09
15  metric  Tomcat02            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory      4  3.229293e+09  3.105422e+09  2.850080e+09  3.230295e+09
16  metric  Tomcat01            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory      5  3.091770e+09  2.842775e+09  2.485992e+09  3.094522e+09
17  metric  dockerA2  Container-DOCKER_CONTAINER_94eca4f96efe--bcou-...      3  3.072000e+09  3.072000e+09  3.072000e+09  3.072000e+09
18  metric  dockerB1  Container-DOCKER_CONTAINER_b30097144a13--bcou-...      2  3.072000e+09  3.072000e+09  3.072000e+09  3.072000e+09
19  metric  dockerA1  Container-DOCKER_CONTAINER_b6760337dc49--bcou-...      1  3.072000e+09  3.072000e+09  3.072000e+09  3.072000e+09,          cmdb_id  count
0        Mysql02   4120
1        Mysql01   4102
2        Redis01   2256
3        Redis02   2248
4       Tomcat04   1655
..           ...    ...
24  ServiceTest3    112
25      dockerA2     32
26      dockerB1     12
27      dockerB2     10
28      dockerA1      8

[29 rows x 2 columns],      cmdb_id  count
0       MG02   1146
1       MG01   1134
2       IG02    450
3       IG01    450
4   Tomcat01    450
5   Tomcat02    450
6   Tomcat03    450
7   Tomcat04    450
8   dockerB2    450
9   dockerB1    450
10  dockerA2    420
11  dockerA1    420,     cmdb_id  count
0  Tomcat02     67
1  Tomcat04     65
2  Tomcat03     64
3  Tomcat01     63
4  apache01     62
5  apache02     60
6   Mysql01      8, Empty DataFrame
Columns: [cmdb_id, count]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse previously loaded DataFrames: metric, trace, log_df (assumed in kernel)
# Ensure ts_dt exists and is timezone-aware Asia/Shanghai; if not, create it
tz = pytz.timezone('Asia/Shanghai')
for df in (metric, trace, log_df):
    if 'ts_dt' not in df.columns:
        df['ts_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        # Ensure tz and type
        if df['ts_dt'].dt.tz is None:
            df['ts_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        else:
            df['ts_dt'] = df['ts_dt'].dt.tz_convert(tz)

# Define allowed root-cause components
allowed_cmdb = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# 1) Compute global P95 thresholds per (source, cmdb_id, name) using full timeseries (restricted to allowed components)
# Metric
metric_filt = metric[metric['cmdb_id'].isin(allowed_cmdb)].copy()
metric_p95 = (
    metric_filt
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95))
)
metric_p95.rename(columns={'kpi_name': 'name'}, inplace=True)
metric_p95['source'] = 'metric'
metric_p95 = metric_p95[['source','cmdb_id','name','p95']]

# Trace
trace_filt = trace[trace['cmdb_id'].isin(allowed_cmdb)].copy()
trace_p95 = (
    trace_filt
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95))
)
trace_p95.rename(columns={'trace_name': 'name'}, inplace=True)
trace_p95['source'] = 'trace'
trace_p95 = trace_p95[['source','cmdb_id','name','p95']]

# Log
log_filt = log_df[log_df['cmdb_id'].isin(allowed_cmdb)].copy()
log_p95 = (
    log_filt
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95))
)
log_p95.rename(columns={'log_name': 'name'}, inplace=True)
log_p95['source'] = 'log'
log_p95 = log_p95[['source','cmdb_id','name','p95']]

# Combine into global_thresholds
global_thresholds = pd.concat([metric_p95, trace_p95, log_p95], ignore_index=True)

# 2) Filter raw rows to the incident window (UTC+8)
start_str = "2021-03-07 06:30:00"
end_str   = "2021-03-07 07:00:00"
start_ts = tz.localize(pd.to_datetime(start_str))
end_ts   = tz.localize(pd.to_datetime(end_str))

metric_win = metric_filt[(metric_filt['ts_dt'] >= start_ts) & (metric_filt['ts_dt'] <= end_ts)].copy()
trace_win  = trace_filt[(trace_filt['ts_dt'] >= start_ts) & (trace_filt['ts_dt'] <= end_ts)].copy()
log_win    = log_filt[(log_filt['ts_dt'] >= start_ts) & (log_filt['ts_dt'] <= end_ts)].copy()

# Normalize column names to merge with thresholds: source, cmdb_id, name, value, ts_dt
metric_win = metric_win.rename(columns={'kpi_name':'name'})
metric_win['source'] = 'metric'
trace_win = trace_win.rename(columns={'trace_name':'name'})
trace_win['source'] = 'trace'
log_win = log_win.rename(columns={'log_name':'name'})
log_win['source'] = 'log'

common_cols = ['source','cmdb_id','name','value','ts_dt']
metric_win = metric_win[common_cols]
trace_win  = trace_win[common_cols]
log_win    = log_win[common_cols]

# 3) Mark anomaly points where value >= global P95 (join with global_thresholds)
# Combine window rows
window_all = pd.concat([metric_win, trace_win, log_win], ignore_index=True)

# Merge thresholds
window_with_thresh = window_all.merge(global_thresholds, on=['source','cmdb_id','name'], how='left')

# If threshold missing (no historical data), set p95 to NaN -> these cannot be marked as anomalies
window_with_thresh['is_anomaly'] = False
mask = (~window_with_thresh['p95'].isna()) & (window_with_thresh['value'] >= window_with_thresh['p95'])
window_with_thresh.loc[mask, 'is_anomaly'] = True

# Save full anomaly rows table (all window rows with threshold and is_anomaly) to anomalies_full
anomalies_full = window_with_thresh.copy()

# 4) For each group that has anomalies, produce compact row
anoms = anomalies_full[anomalies_full['is_anomaly']].copy()
if not anoms.empty:
    grouped = anoms.groupby(['source','cmdb_id','name'], as_index=False).agg(
        earliest_anomaly_timestamp=('ts_dt', 'min'),
        anomaly_count_in_window=('value', 'count'),
        max_value_in_window=('value', 'max'),
        p95_threshold=('p95', 'first')  # same per group
    )
    # Severity ratio calculation
    def calc_sev(row):
        p = row['p95_threshold']
        if pd.isna(p) or p == 0:
            return np.nan
        return (row['max_value_in_window'] - p) / p
    grouped['severity_ratio'] = grouped.apply(calc_sev, axis=1)
    # Format earliest timestamp as string in Asia/Shanghai
    grouped['earliest_anomaly_timestamp'] = grouped['earliest_anomaly_timestamp'].dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
    # Sort by earliest_anomaly_timestamp asc then severity_ratio desc
    grouped = grouped.sort_values(['earliest_anomaly_timestamp', 'severity_ratio'], ascending=[True, False]).reset_index(drop=True)
    compact_table = grouped[['source','cmdb_id','name','earliest_anomaly_timestamp','anomaly_count_in_window','max_value_in_window','p95_threshold','severity_ratio']].head(20)
else:
    # No anomalies found
    compact_table = pd.DataFrame(columns=['source','cmdb_id','name','earliest_anomaly_timestamp','anomaly_count_in_window','max_value_in_window','p95_threshold','severity_ratio'])

# 5) total number of anomaly groups
total_anomaly_groups = compact_table.shape[0]

# Display compact table (top 20) and confirm the number of total anomaly groups found
compact_table, total_anomaly_groups

```
Out[2]:
```
summary = None
try:
    # Use existing kernel variables from prior steps
    df = compact_table  # compact_table created in previous cell
    # Basic stats
    total_groups = int(df.shape[0])
    sources = df['source'].unique().tolist()
    earliest_time = df['earliest_anomaly_timestamp'].min() if total_groups>0 else None
    # Components and counts
    comp_counts = df['cmdb_id'].value_counts().to_dict()
    # Top 3 by severity_ratio
    top3 = df.sort_values('severity_ratio', ascending=False).head(3)[['cmdb_id','name','severity_ratio','earliest_anomaly_timestamp']].copy()
    top3['severity_ratio'] = top3['severity_ratio'].round(4)
    top3_rows = top3.to_dict('records')
    
    # Build plain-English summary
    lines = []
    lines.append(f"Total anomaly groups found: {total_groups}.")
    lines.append(f"Sources involved: {', '.join(sources)}.")
    if earliest_time is not None:
        lines.append(f"Earliest anomaly timestamp (UTC+8): {earliest_time}.")
    # Components summary
    comp_list_str = ", ".join([f"{k} ({v})" for k,v in comp_counts.items()])
    lines.append(f"Affected components (counts in compact table): {comp_list_str}.")
    # Top 3 severe groups
    if top3_rows:
        lines.append("Top 3 most severe anomaly groups (by severity_ratio):")
        for i, r in enumerate(top3_rows, start=1):
            lines.append(f" {i}) {r['cmdb_id']} — {r['name']} | severity_ratio={r['severity_ratio']} | earliest={r['earliest_anomaly_timestamp']}")
    # Where results are stored
    lines.append("Full global thresholds are stored in kernel variable: global_thresholds.")
    lines.append("All full anomaly rows for the window are stored in kernel variable: anomalies_full.")
    
    summary = "\n".join(lines)
except Exception as e:
    summary = f"Could not build summary due to error: {e}"

summary

The original code execution output of IPython Kernel is also provided below for reference:

(    source   cmdb_id                                               name earliest_anomaly_timestamp  anomaly_count_in_window  max_value_in_window  p95_threshold  severity_ratio
0   metric   Mysql01    Mysql-MySQL_3306_Innodb buffer pool pages dirty   2021-03-07 06:30:00+0800                        2            84.000000       4.000000       20.000000
1   metric   Mysql01         Mysql-MySQL_3306_Innodb log write requests   2021-03-07 06:30:00+0800                        2             1.900000       0.283300        5.706671
2   metric   Mysql02                 Mysql-MySQL_3306_Handler Read Next   2021-03-07 06:30:00+0800                        2             8.033300       3.366700        1.386105
3   metric   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKRead   2021-03-07 06:30:00+0800                        1             0.533333       0.233333        1.285716
4   metric   Mysql02                  Mysql-MySQL_3306_Handler Read Rnd   2021-03-07 06:30:00+0800                        2             0.283300       0.133300        1.125281
5   metric   Mysql02                        Mysql-MySQL_3306_Sort Range   2021-03-07 06:30:00+0800                        2             0.283300       0.133300        1.125281
6   metric   Mysql01                  Mysql-MySQL_3306_Binlog Cache Use   2021-03-07 06:30:00+0800                        3             0.516700       0.250000        1.066800
7   metric   Mysql02                     Mysql-MySQL_3306_Opened Tables   2021-03-07 06:30:00+0800                        3             0.131100       0.066700        0.965517
8   metric   Mysql02           Mysql-MySQL_3306_Table open cache misses   2021-03-07 06:30:00+0800                        3             0.131100       0.066700        0.965517
9   metric   Mysql01             Mysql-MySQL_3306_Innodb os log written   2021-03-07 06:30:00+0800                        2          1237.333300     648.533300        0.907895
10  metric  Tomcat02                    OSLinux-CPU_CPU-3_SingleCpuidle   2021-03-07 06:30:00+0800                        1            27.546100      15.052100        0.830050
11  metric   Mysql01  Mysql-MySQL_3306_Innodb buffer pool write requ...   2021-03-07 06:30:00+0800                        2             5.816700       3.383300        0.719239
12  metric      MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps   2021-03-07 06:30:00+0800                        1             0.083300       0.049980        0.666667
13  metric      MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead   2021-03-07 06:30:00+0800                        1             1.466700       0.880020        0.666667
14  metric   Mysql01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKWrite   2021-03-07 06:30:00+0800                        1           260.800000     162.333333        0.606571
15  metric  Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps   2021-03-07 06:30:00+0800                        1             3.000000       2.000000        0.500000
16  metric   Mysql02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...   2021-03-07 06:30:00+0800                       31             0.600000       0.500000        0.200000
17  metric   Mysql01                Mysql-MySQL_3306_Handler Read First   2021-03-07 06:30:00+0800                        4             0.416700       0.350000        0.190571
18  metric   Mysql01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite   2021-03-07 06:30:00+0800                        1            14.533333      12.293310        0.182215
19  metric   Mysql01                       Mysql-MySQL_3306_Select Scan   2021-03-07 06:30:00+0800                        4             0.533300       0.466700        0.142704, 20)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing 'trace' dataframe if present; otherwise load from CSV
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")

# Ensure timezone-aware ts_dt in Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if 'ts_dt' not in trace.columns:
    trace['ts_dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and converted
    trace['ts_dt'] = pd.to_datetime(trace['ts_dt'])
    if trace['ts_dt'].dt.tz is None:
        trace['ts_dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        trace['ts_dt'] = trace['ts_dt'].dt.tz_convert(tz)

# Allowed root-cause components
allowed_cmdb = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# 1) Compute global P95 per (cmdb_id, trace_name) using full trace timeseries (restricted to allowed components)
trace_filt_all = trace[trace['cmdb_id'].isin(allowed_cmdb)].copy()
trace_global_thresholds = (
    trace_filt_all
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95))
    .rename(columns={'trace_name': 'name'})
)
trace_global_thresholds['source'] = 'trace'
trace_global_thresholds = trace_global_thresholds[['source','cmdb_id','name','p95']]

# 2) Filter trace rows to the incident window (UTC+8)
start_str = "2021-03-07 06:30:00"
end_str   = "2021-03-07 07:00:00"
start_ts = tz.localize(pd.to_datetime(start_str))
end_ts   = tz.localize(pd.to_datetime(end_str))

trace_window = trace_filt_all[(trace_filt_all['ts_dt'] >= start_ts) & (trace_filt_all['ts_dt'] <= end_ts)].copy()

# Normalize columns for merging
trace_window = trace_window.rename(columns={'trace_name':'name'})
trace_window['source'] = 'trace'
trace_window = trace_window[['source','cmdb_id','name','value','ts_dt']]

# 3) Mark anomaly rows where value >= group P95 (merge thresholds)
trace_window_with_thresh = trace_window.merge(trace_global_thresholds, on=['source','cmdb_id','name'], how='left')
trace_window_with_thresh['is_anomaly'] = False
mask = (~trace_window_with_thresh['p95'].isna()) & (trace_window_with_thresh['value'] >= trace_window_with_thresh['p95'])
trace_window_with_thresh.loc[mask, 'is_anomaly'] = True

# 4) For groups that have anomalies, produce compact rows
trace_anomalies_full = trace_window_with_thresh[trace_window_with_thresh['is_anomaly']].copy()  # save full anomaly rows (kernel variable)

if not trace_anomalies_full.empty:
    grouped = trace_anomalies_full.groupby(['source','cmdb_id','name'], as_index=False).agg(
        earliest_anomaly_timestamp=('ts_dt', 'min'),
        anomaly_count_in_window=('value', 'count'),
        max_value_in_window=('value', 'max'),
        p95_threshold=('p95', 'first')
    )
    # severity_ratio calculation with null when p95==0 or NaN
    def sev(row):
        p = row['p95_threshold']
        if pd.isna(p) or p == 0:
            return np.nan
        return (row['max_value_in_window'] - p) / p
    grouped['severity_ratio'] = grouped.apply(sev, axis=1)
    # Format earliest timestamp as string with timezone offset
    grouped['earliest_anomaly_timestamp'] = grouped['earliest_anomaly_timestamp'].dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
    # Sort as requested
    trace_compact_table = grouped.sort_values(['earliest_anomaly_timestamp', 'severity_ratio'], ascending=[True, False])[
        ['source','cmdb_id','name','earliest_anomaly_timestamp','anomaly_count_in_window','max_value_in_window','p95_threshold','severity_ratio']
    ].head(20).reset_index(drop=True)
else:
    trace_compact_table = pd.DataFrame(columns=['source','cmdb_id','name','earliest_anomaly_timestamp','anomaly_count_in_window','max_value_in_window','p95_threshold','severity_ratio'])

# 5) Total number of trace anomaly groups found
total_trace_anomaly_groups = trace_compact_table.shape[0]

# Display compact table and total count (do not print full trace_anomalies_full)
trace_compact_table, total_trace_anomaly_groups

```
Out[3]:
```
summary = ""
try:
    # Use existing kernel variables created previously
    df = trace_compact_table  # compact table (top 20) from previous step
    total = int(total_trace_anomaly_groups)
    
    # Basic stats
    earliest = df['earliest_anomaly_timestamp'].min() if not df.empty else "N/A"
    comps = df['cmdb_id'].value_counts().to_dict()
    top_comps = ", ".join([f"{k} ({v})" for k,v in list(comps.items())[:5]]) if comps else "None"
    
    # Top 3 by severity_ratio
    top3 = df.sort_values('severity_ratio', ascending=False).head(3)
    top3_lines = []
    for _, r in top3.iterrows():
        sev = r['severity_ratio']
        sev_str = "null" if pd.isna(sev) else f"{sev:.3f}"
        top3_lines.append(f"{r['cmdb_id']} — {r['name']} | severity_ratio={sev_str} | earliest={r['earliest_anomaly_timestamp']}")
    
    # Build plain-English summary
    lines = []
    lines.append(f"Trace anomaly summary (incident window 2021-03-07 06:30–07:00 UTC+8):")
    lines.append(f"- Total trace anomaly groups found: {total}.")
    lines.append(f"- Earliest anomaly timestamp (UTC+8) in results: {earliest}.")
    lines.append(f"- Top affected components (counts in compact table): {top_comps}.")
    if top3_lines:
        lines.append("- Top 3 most severe trace anomaly groups:")
        for i, l in enumerate(top3_lines, 1):
            lines.append(f"  {i}) {l}")
    lines.append("- Full per-group thresholds stored in kernel variable: trace_global_thresholds.")
    lines.append("- All trace anomaly rows for the window saved to kernel variable: trace_anomalies_full.")
    
    summary = "\n".join(lines)
except Exception as e:
    summary = f"Could not produce summary due to error: {e}"

summary

The original code execution output of IPython Kernel is also provided below for reference:

(   source   cmdb_id                               name earliest_anomaly_timestamp  anomaly_count_in_window  max_value_in_window  p95_threshold  severity_ratio
0   trace      IG01     trace.to_Tomcat04.duration_p95   2021-03-07 06:31:00+0800                        2             0.799900       0.486542        0.644050
1   trace  Tomcat04       trace.from_IG01.duration_p95   2021-03-07 06:31:00+0800                        2             0.799900       0.486542        0.644050
2   trace      MG01     trace.to_dockerA2.duration_p95   2021-03-07 06:31:00+0800                        2             0.414350       0.290180        0.427907
3   trace      MG01    trace.to_dockerB2.duration_mean   2021-03-07 06:31:00+0800                        2             0.138500       0.099338        0.394226
4   trace      MG01  trace.from_Tomcat04.duration_mean   2021-03-07 06:31:00+0800                        2             0.157417       0.119066        0.322096
5   trace  Tomcat04        trace.to_MG01.duration_mean   2021-03-07 06:31:00+0800                        2             0.157417       0.119066        0.322096
6   trace      MG01   trace.from_Tomcat04.duration_p95   2021-03-07 06:31:00+0800                        2             0.621250       0.499270        0.244317
7   trace  Tomcat04         trace.to_MG01.duration_p95   2021-03-07 06:31:00+0800                        2             0.621250       0.499270        0.244317
8   trace      MG01    trace.to_dockerA2.duration_mean   2021-03-07 06:31:00+0800                        2             0.091000       0.075809        0.200386
9   trace      MG02    trace.to_dockerB1.duration_mean   2021-03-07 06:31:00+0800                        2             0.157364       0.132098        0.191268
10  trace      IG02    trace.to_Tomcat01.duration_mean   2021-03-07 06:31:00+0800                        2             0.131727       0.110896        0.187840
11  trace  Tomcat01      trace.from_IG02.duration_mean   2021-03-07 06:31:00+0800                        2             0.131727       0.110896        0.187840
12  trace      MG02   trace.from_dockerB1.duration_p95   2021-03-07 06:31:00+0800                        2             0.015000       0.012637        0.186944
13  trace  Tomcat01           trace.self.duration_mean   2021-03-07 06:31:00+0800                        2             0.015826       0.014720        0.075103
14  trace      MG01   trace.from_dockerB1.duration_p95   2021-03-07 06:32:00+0800                        2             0.353700       0.208077        0.699847
15  trace      MG01  trace.from_dockerB1.duration_mean   2021-03-07 06:32:00+0800                        2             0.057292       0.051191        0.119173
16  trace      MG02   trace.from_Tomcat03.duration_p95   2021-03-07 06:33:00+0800                        2             1.043600       0.504070        1.070347
17  trace  Tomcat03         trace.to_MG02.duration_p95   2021-03-07 06:33:00+0800                        2             1.043600       0.504070        1.070347
18  trace      MG01   trace.from_Tomcat03.duration_p95   2021-03-07 06:33:00+0800                        2             1.119500       0.560375        0.997769
19  trace  Tomcat03         trace.to_MG01.duration_p95   2021-03-07 06:33:00+0800                        2             1.119500       0.560375        0.997769, 20)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
import numpy as np

# Reuse anomalies_full from kernel (assumed present)
# Ensure tz
tz = pytz.timezone('Asia/Shanghai')

# Filter rows: source == 'metric' and cmdb_id == 'Mysql01' and is_anomaly == True
try:
    filtered = anomalies_full[
        (anomalies_full['source'] == 'metric') &
        (anomalies_full['cmdb_id'] == 'Mysql01') &
        (anomalies_full['is_anomaly'] == True)
    ].copy()
except NameError:
    # If anomalies_full not defined, create empty DataFrame
    filtered = pd.DataFrame(columns=['source','cmdb_id','name','value','ts_dt','p95','is_anomaly'])

# Ensure ts_dt exists and is tz-aware Asia/Shanghai
if 'ts_dt' in filtered.columns and not filtered['ts_dt'].empty:
    # convert to datetime if needed
    filtered['ts_dt'] = pd.to_datetime(filtered['ts_dt'])
    if filtered['ts_dt'].dt.tz is None:
        filtered['ts_dt'] = filtered['ts_dt'].dt.tz_localize(pytz.UTC).dt.tz_convert(tz)
    else:
        filtered['ts_dt'] = filtered['ts_dt'].dt.tz_convert(tz)

# Create timestamp column as epoch seconds (UTC) derived from ts_dt for compact raw output
if 'ts_dt' in filtered.columns and not filtered['ts_dt'].empty:
    # convert to UTC then to epoch seconds
    filtered['timestamp'] = (filtered['ts_dt'].dt.tz_convert(pytz.UTC).view('int64') // 10**9).astype('int64')
else:
    filtered['timestamp'] = pd.Series(dtype='int64')

# 1) Raw anomaly rows with columns: timestamp, cmdb_id, name, value sorted by timestamp asc limited to 50
if not filtered.empty:
    mysql01_anomaly_rows = filtered[['timestamp','cmdb_id','name','value']].sort_values('timestamp', ascending=True).head(50).reset_index(drop=True)
else:
    mysql01_anomaly_rows = pd.DataFrame(columns=['timestamp','cmdb_id','name','value'])

# 2) Per-KPI summary for these Mysql01 anomaly rows
if not filtered.empty:
    summary = (
        filtered
        .groupby('name', as_index=False)
        .agg(
            anomaly_count_in_window=('value', 'count'),
            earliest_anomaly_timestamp=('ts_dt', 'min'),
            max_value_in_window=('value', 'max')
        )
    )
    # Format earliest_anomaly_timestamp as Asia/Shanghai string
    summary['earliest_anomaly_timestamp'] = summary['earliest_anomaly_timestamp'].dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
    # Limit to all KPIs for Mysql01 (no more than 50 rows)
    mysql01_anomaly_summary = summary.head(50).reset_index(drop=True)
else:
    mysql01_anomaly_summary = pd.DataFrame(columns=['name','anomaly_count_in_window','earliest_anomaly_timestamp','max_value_in_window'])

# Save variables to kernel as requested
# (They are already assigned: mysql01_anomaly_rows, mysql01_anomaly_summary)

# Display only the two compact outputs
mysql01_anomaly_rows, mysql01_anomaly_summary

```
Out[4]:
```
summary = ""
try:
    # Reuse kernel variables created earlier
    raw = mysql01_anomaly_rows
    summary_df = mysql01_anomaly_summary

    raw_count = int(raw.shape[0])
    kpi_count = int(summary_df.shape[0])

    if kpi_count > 0:
        earliest = summary_df['earliest_anomaly_timestamp'].min()
        # top KPIs by max_value_in_window
        top_k = summary_df.sort_values('max_value_in_window', ascending=False).head(5)
        top_lines = []
        for _, r in top_k.iterrows():
            name = r['name']
            mv = r['max_value_in_window']
            ets = r['earliest_anomaly_timestamp']
            top_lines.append(f"- {name}: max={mv} (earliest anomaly at {ets})")
        top_str = "\n".join(top_lines)
    else:
        earliest = "N/A"
        top_str = "None"

    summary = (
        f"Mysql01 anomaly extraction summary:\n\n"
        f"- Raw anomaly rows returned: {raw_count} (saved in kernel variable 'mysql01_anomaly_rows').\n"
        f"- Distinct KPIs with anomalies: {kpi_count} (saved in kernel variable 'mysql01_anomaly_summary').\n"
        f"- Earliest anomaly timestamp (UTC+8) among these KPIs: {earliest}.\n\n"
        f"- Top KPIs by observed max value in the window:\n{top_str}\n\n"
        f"Notes:\n"
        f"- Many KPI max values are zero; only a few KPIs show notable non-zero peaks (listed above).\n"
        f"- Use the kernel variables 'mysql01_anomaly_rows' and 'mysql01_anomaly_summary' for full details."
    )
except Exception as e:
    summary = f"Could not build summary due to error: {e}"

summary

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp  cmdb_id                                         name   value
0   1615069800  Mysql01             Mysql-MySQL_3306_Aborted Clients  0.0000
1   1615069800  Mysql01            Mysql-MySQL_3306_Aborted Connects  0.0000
2   1615069800  Mysql01       Mysql-MySQL_3306_Binlog Cache Disk Use  0.0000
3   1615069800  Mysql01            Mysql-MySQL_3306_Binlog Cache Use  0.5167
4   1615069800  Mysql01  Mysql-MySQL_3306_Binlog stmt cache disk use  0.0000
..         ...      ...                                          ...     ...
45  1615069800  Mysql01                  Mysql-MySQL_3306_Com Insert  0.2667
46  1615069800  Mysql01          Mysql-MySQL_3306_Com Replace Select  0.0000
47  1615069800  Mysql01                 Mysql-MySQL_3306_Com Replace  0.0000
48  1615069800  Mysql01           Mysql-MySQL_3306_Created Tmp Files  0.0000
49  1615069800  Mysql01     Mysql-MySQL_3306_Created Tmp Disk Tables  0.0000

[50 rows x 4 columns],                                                  name  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window
0                    Mysql-MySQL_3306_Aborted Clients                       21   2021-03-07 06:30:00+0800               0.0000
1                   Mysql-MySQL_3306_Aborted Connects                       21   2021-03-07 06:30:00+0800               0.0000
2              Mysql-MySQL_3306_Binlog Cache Disk Use                       21   2021-03-07 06:30:00+0800               0.0000
3                   Mysql-MySQL_3306_Binlog Cache Use                        3   2021-03-07 06:30:00+0800               0.5167
4         Mysql-MySQL_3306_Binlog stmt cache disk use                       21   2021-03-07 06:30:00+0800               0.0000
..                                                ...                      ...                        ...                  ...
45     Mysql-MySQL_3306_Innodb buffer pool read ahead                       21   2021-03-07 06:30:00+0800               0.0000
46  Mysql-MySQL_3306_Innodb buffer pool read ahead...                       21   2021-03-07 06:30:00+0800               0.0000
47  Mysql-MySQL_3306_Innodb buffer pool read ahead...                       21   2021-03-07 06:30:00+0800               0.0000
48  Mysql-MySQL_3306_Innodb buffer pool read requests                        2   2021-03-07 06:36:00+0800              36.2000
49          Mysql-MySQL_3306_Innodb buffer pool reads                       21   2021-03-07 06:30:00+0800               0.0000

[50 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```